### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.models import load_model

In [2]:
#load the dataset
data=pd.read_csv('../../artifacts/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data.drop(columns=['RowNumber',"CustomerId","Surname"],inplace=True, axis=1)

In [3]:
# load the preprocessor
preprocessor_path="../../artifacts/classification/preprocessor.pkl"
with open(preprocessor_path,'rb') as file:
    preprocessor=pickle.load(file)

In [4]:
# load the model
model_path="../../artifacts/classification/model.h5"
model=load_model(model_path)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(data.drop('Exited',axis=1),data['Exited'])

In [7]:
X_train=preprocessor.transform(X_train)
X_test=preprocessor.transform(X_test)

In [9]:
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import GridSearchCV

In [10]:
# define a function to hypertune the ann model

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [11]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [12]:
# Define the grid search parameters
param_grid = {
    'neurons': [8,16, 32, 64],
    'layers': [1, 2],
    'epochs': [20,40]
}

In [13]:
# perform grid seach cv
grid=GridSearchCV(estimator=model,param_grid=param_grid, n_jobs=-1,cv=3)
grid_result=grid.fit(X_train,y_train)


Epoch 1/40


235/235 [==============================] - 5s 3ms/step - loss: 0.4990 - accuracy: 0.7824
Epoch 2/40
235/235 [==============================] - 1s 2ms/step - loss: 0.4121 - accuracy: 0.8221
Epoch 3/40
235/235 [==============================] - 1s 2ms/step - loss: 0.3759 - accuracy: 0.8439
Epoch 4/40
235/235 [==============================] - 1s 2ms/step - loss: 0.3557 - accuracy: 0.8533
Epoch 5/40
235/235 [==============================] - 0s 2ms/step - loss: 0.3459 - accuracy: 0.8587
Epoch 6/40
235/235 [==============================] - 0s 2ms/step - loss: 0.3417 - accuracy: 0.8611
Epoch 7/40
235/235 [==============================] - 0s 2ms/step - loss: 0.3390 - accuracy: 0.8595
Epoch 8/40
235/235 [==============================] - 1s 2ms/step - loss: 0.3370 - accuracy: 0.8612
Epoch 9/40
235/235 [==============================] - 0s 2ms/step - loss: 0.3359 - accuracy: 0.8609
Epoch 10/40
235/235 [==============================] - 0s 2ms/step - loss: 0.3336 - accuracy: 0.8

In [14]:
# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best: 0.860400 using {'epochs': 40, 'layers': 2, 'neurons': 16}
